# Notebook 04 — Inference Only (v2)

This notebook **does not build indices**. It only loads the measurement *bundle* produced by Notebook 05 and runs inference (effect sizes + uncertainty + bootstrap sign stability + arc tests).

In [1]:
# ==============================
# SECTION 0 — Setup & bundle discovery
# ==============================
from __future__ import annotations

from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

def normalize_id(x) -> str:
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

# Find project root by walking up until we find both 'results' and 'data' directories
def find_project_root(start: Path) -> Path:
    """Walk up directory tree to find project root (contains both 'results' and 'data' directories)."""
    current = start.resolve()
    while current != current.parent:
        results_dir = current / "results"
        data_dir = current / "data"
        if results_dir.exists() and results_dir.is_dir() and data_dir.exists() and data_dir.is_dir():
            return current
        current = current.parent
    # Fallback: use environment variable
    env_root = os.environ.get("ROMANCE_ROOT", "")
    if env_root:
        return Path(env_root).expanduser().resolve()
    return Path.cwd().resolve()

# Try multiple strategies to find project root
PROJECT_ROOT = None

# Strategy 1: Use environment variable
env_root = os.environ.get("ROMANCE_ROOT", "")
if env_root and Path(env_root).expanduser().exists():
    PROJECT_ROOT = Path(env_root).expanduser().resolve()

# Strategy 2: Walk up from current working directory
if PROJECT_ROOT is None or not (PROJECT_ROOT / "results").exists():
    candidate = find_project_root(Path.cwd())
    if (candidate / "results").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate

# Final fallback: use known absolute path
if PROJECT_ROOT is None or not (PROJECT_ROOT / "results").exists():
    known_path = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
    if known_path.exists() and (known_path / "results").exists() and (known_path / "data").exists():
        PROJECT_ROOT = known_path

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not determine PROJECT_ROOT. Please set ROMANCE_ROOT environment variable.")

PROJECT_ROOT = PROJECT_ROOT.resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
DATA_DIR = PROJECT_ROOT / "data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("DATA_DIR:", DATA_DIR)

# Point to composite_measurement bundle and audit directories
_bundle_env = os.environ.get("MEASUREMENT_BUNDLE_DIR", "").strip()
if _bundle_env:
    BUNDLE_DIR = Path(_bundle_env).expanduser().resolve()
else:
    # Default to composite_measurement/bundle
    BUNDLE_DIR = RESULTS_DIR / "stage10_correlation_analysis" / "composite_measurement" / "bundle"
    if not BUNDLE_DIR.exists():
        raise FileNotFoundError(f"Bundle directory not found: {BUNDLE_DIR}. Please set MEASUREMENT_BUNDLE_DIR or ensure composite_measurement/bundle exists.")

# Audit directory (for files like book_indices_z.csv)
AUDIT_DIR_SOURCE = RESULTS_DIR / "stage10_correlation_analysis" / "composite_measurement" / "audit"
if not AUDIT_DIR_SOURCE.exists():
    raise FileNotFoundError(f"Audit directory not found: {AUDIT_DIR_SOURCE}")

print("Using BUNDLE_DIR:", BUNDLE_DIR)
print("Using AUDIT_DIR_SOURCE:", AUDIT_DIR_SOURCE)


PROJECT_ROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
RESULTS_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results
DATA_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/data
Using BUNDLE_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/bundle
Using AUDIT_DIR_SOURCE: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/audit


In [2]:
# ==============================
# SECTION 1 — Load bundle tables
# ==============================
def read_csv(name: str, check_audit: bool = False) -> pd.DataFrame:
    """Read CSV file, checking bundle directory first, then audit if requested."""
    # Try bundle directory first
    path = BUNDLE_DIR / name
    if path.exists():
        return pd.read_csv(path)
    # If not found and check_audit is True, try audit directory
    if check_audit:
        path = AUDIT_DIR_SOURCE / name
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(f"Missing file: {name} (checked BUNDLE_DIR={BUNDLE_DIR} and AUDIT_DIR_SOURCE={AUDIT_DIR_SOURCE if check_audit else 'N/A'})")

# Core bundle files (in bundle directory)
composite_registry = read_csv("composite_registry.csv")
composite_diagnostics = read_csv("composite_diagnostics.csv")
pipeline_audit = read_csv("pipeline_audit.csv")
core_variables = read_csv("core_variables.csv")

# Book indices are in audit directory
book_indices_z = read_csv("book_indices_z.csv", check_audit=True)
book_indices_raw = read_csv("book_indices_raw.csv", check_audit=True)

# Optional bundle files
arc_contrasts_sum = None
pca_scores_long = None
prob_contract_book = None
prob_contract_segment = None
segment_indices_raw = None
segment_indices_z = None

# Check for optional files in bundle directory
for opt in ["arc_contrasts_sum.csv", "composite_pca_scores_long.csv", "prob_contract_book.csv", "prob_contract_segment.csv"]:
    p = BUNDLE_DIR / opt
    if p.exists():
        if opt == "arc_contrasts_sum.csv":
            arc_contrasts_sum = pd.read_csv(p)
        elif opt == "composite_pca_scores_long.csv":
            pca_scores_long = pd.read_csv(p)
        elif opt == "prob_contract_book.csv":
            prob_contract_book = pd.read_csv(p)
        elif opt == "prob_contract_segment.csv":
            prob_contract_segment = pd.read_csv(p)

# Check for segment indices in audit directory
for opt in ["segment_indices_raw.csv", "segment_indices_z.csv"]:
    p = AUDIT_DIR_SOURCE / opt
    if p.exists():
        if opt == "segment_indices_raw.csv":
            segment_indices_raw = pd.read_csv(p)
        elif opt == "segment_indices_z.csv":
            segment_indices_z = pd.read_csv(p)

print("Loaded bundle tables:")
print("  composite_registry:", composite_registry.shape)
print("  composite_diagnostics:", composite_diagnostics.shape)
print("  pipeline_audit:", pipeline_audit.shape)
print("  core_variables:", core_variables.shape)
print("  book_indices_z:", book_indices_z.shape)
print("  book_indices_raw:", book_indices_raw.shape)
print("  arc_contrasts_sum:", None if arc_contrasts_sum is None else arc_contrasts_sum.shape)
print("  pca_scores_long:", None if pca_scores_long is None else pca_scores_long.shape)
print("  segment_indices_raw:", None if segment_indices_raw is None else segment_indices_raw.shape)
print("  segment_indices_z:", None if segment_indices_z is None else segment_indices_z.shape)

# Contract checks (these should be clean if topic probs were normalized upstream)
if prob_contract_book is not None:
    s = prob_contract_book["sum_prob"]
    print("\nBook prob sum contract (from bundle): min/median/max =", float(s.min()), float(s.median()), float(s.max()))
if prob_contract_segment is not None:
    s = prob_contract_segment["sum_prob"]
    print("Segment prob sum contract (from bundle): min/median/max =", float(s.min()), float(s.median()), float(s.max()))


Loaded bundle tables:
  composite_registry: (26, 30)
  composite_diagnostics: (26, 9)
  pipeline_audit: (26, 17)
  core_variables: (17, 9)
  book_indices_z: (92, 40)
  book_indices_raw: (92, 40)
  arc_contrasts_sum: (92, 53)
  pca_scores_long: (2392, 4)
  segment_indices_raw: (276, 28)
  segment_indices_z: (276, 28)

Book prob sum contract (from bundle): min/median/max = 1.0 1.0 1.0000000000000002
Segment prob sum contract (from bundle): min/median/max = 0.9999999999999998 1.0 1.0000000000000002


## OUTPUT MANAGEMENT — clean results folder structure

This notebook writes **CSV tables** for audit/appendix and **publication-ready figures** into a clearly named subfolder under `results/`.

In [ ]:
# ==============================
# OUTPUT MANAGEMENT
# ==============================
from datetime import datetime
from pathlib import Path
import json
import re

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

# Name the run folder using the bundle parent (helps trace provenance)
bundle_name = BUNDLE_DIR.parent.name if BUNDLE_DIR.name == "bundle" else BUNDLE_DIR.name
RUN_NAME = f"inference_{RUN_TS}__{bundle_name}"

OUTROOT = RESULTS_DIR / "stage10_correlation_analysis" / "hypothesis_testing" / RUN_NAME
AUDIT_DIR = OUTROOT / "audit_csv"
APPENDIX_DIR = OUTROOT / "appendix_csv"
FIG_DIR = OUTROOT / "figures"
LOG_DIR = OUTROOT / "logs"

for d in [AUDIT_DIR, APPENDIX_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def save_df(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def save_fig(fig, path: Path, dpi: int = 300) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")

# Save a run manifest for reproducibility
manifest = {
    "run_name": RUN_NAME,
    "timestamp": RUN_TS,
    "project_root": str(PROJECT_ROOT),
    "bundle_dir": str(BUNDLE_DIR),
    "sentence_df_path": str(SENTENCE_DF_PATH) if "SENTENCE_DF_PATH" in globals() else None,
}
(Path(LOG_DIR) / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("RUN_NAME:", RUN_NAME)
print("OUTROOT:", OUTROOT)
print("AUDIT_DIR:", AUDIT_DIR)
print("APPENDIX_DIR:", APPENDIX_DIR)
print("FIG_DIR:", FIG_DIR)


RUN_NAME: inference_20260112_213754__measurement_v5
OUTROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/inference_runs/inference_20260112_213754__measurement_v5
AUDIT_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/inference_runs/inference_20260112_213754__measurement_v5/audit_csv
APPENDIX_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/inference_runs/inference_20260112_213754__measurement_v5/appendix_csv
FIG_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/inference_runs/inference_20260112_213754__measurement_v5/figures


In [4]:
# ==============================
# SECTION 1.1 — Load metadata (outcomes) from sentence_df (source of truth)
# ==============================
# We keep outcomes separate from measurement to avoid leakage.
SENTENCE_DF_PATH = Path(os.environ.get("SENTENCE_DF_PATH", DATA_DIR / "processed" / "sentence_df_with_topics.parquet"))
if not SENTENCE_DF_PATH.exists():
    raise FileNotFoundError(f"Missing sentence_df: {SENTENCE_DF_PATH}")

sent = pd.read_parquet(SENTENCE_DF_PATH)

# Required columns for outcomes
required = ["book_id", "rating_mean", "rating_count"]
missing = [c for c in required if c not in sent.columns]
if missing:
    raise ValueError(f"sentence_df missing required columns: {missing}. Available: {list(sent.columns)}")

sent = sent.copy()
sent["book_id"] = sent["book_id"].map(normalize_id)

# collapse to book-level metadata (take first non-null)
meta_cols = ["rating_mean", "rating_count", "rating_class", "Author", "Book Title"]
meta_cols = [c for c in meta_cols if c in sent.columns]

meta = (sent.sort_values(["book_id"])
            .groupby("book_id", as_index=False)[meta_cols]
            .first())

# Add log popularity proxy
meta["log_rating_count"] = np.log1p(meta["rating_count"].astype(float))

print("meta:", meta.shape)
print(meta.head())


meta: (92, 7)
     book_id  rating_mean  rating_count rating_class           Author  \
0  104659050         4.06         39826          mid  Catharina_Maura   
1   11266880         3.97         28526          mid         Ann_Cole   
2  123257687         3.89        103954          bad          LJ_Shen   
3  123446478         3.95         28772          mid       Rose_Shain   
4  127305713         4.25         53817         good          LJ_Shen   

                  Book Title  log_rating_count  
0      The Unwanted Marriage         10.592300  
1               I Choose You         10.258606  
2                The Villain         11.551713  
3  Between Love and Loathing         10.267193  
4                 The Hunter         10.893363  


In [5]:
# ==============================
# SECTION 1.2 — Build modeling table (indices + outcomes)
# ==============================
# Normalize ids
for df in [book_indices_z, book_indices_raw]:
    df["book_id"] = df["book_id"].map(normalize_id)

df = book_indices_z.merge(meta, on="book_id", how="inner")
print("Merged df:", df.shape)

# Add PCA scores as alternative predictors for multidimensional composites
if pca_scores_long is not None:
    pca_scores_long["book_id"] = pca_scores_long["book_id"].map(normalize_id)
    wide_pc1 = pca_scores_long.pivot_table(index="book_id", columns="composite_key", values="pc1", aggfunc="mean")
    wide_pc1.columns = [f"{c}__pc1" for c in wide_pc1.columns]
    wide_pc1 = wide_pc1.reset_index()
    df = df.merge(wide_pc1, on="book_id", how="left")
    print("Added PC1 predictors:", wide_pc1.shape)

# Add arc contrasts (end-begin and mid-begin) if available
if arc_contrasts_sum is not None:
    arc_contrasts_sum["book_id"] = arc_contrasts_sum["book_id"].map(normalize_id)
    df = df.merge(arc_contrasts_sum, on="book_id", how="left")
    print("Added arc contrasts:", arc_contrasts_sum.shape)

# Final checks
assert df["book_id"].nunique() == meta["book_id"].nunique(), "Mismatch after merge; investigate missing books."
print("Final modeling df columns:", len(df.columns))


Merged df: (92, 46)
Added PC1 predictors: (92, 27)
Added arc contrasts: (92, 53)
Final modeling df columns: 124


## MACRO-AXES (3–5 higher-level dimensions)

We build a small set of **macro-axes** from CORE predictors to reduce collinearity and improve interpretability.

Each macro-axis is a weighted average of standardized components (z-scored). You can edit the definitions below.

In [6]:
# ==============================
# MACRO-AXES — define 3–5 interpretable dimensions
# ==============================
# Edit these definitions to match your theory. Values are weights (+/-).
# Note: use predictor column names (e.g., 'D_power_wealth_luxury__pc1' if recommended_score=pc1)

MACRO_AXES = {
    # 1) Status/Dominance package (billionaire-romance backbone)
    "AX_status_dominance": {
        "D_power_wealth_luxury__pc1": 1.0,
        "R2_alpha_guarding": 1.0,
    },
    # 2) Emotional payoff / safety (HEA-related affect regulation)
    "AX_payoff_safety": {
        "A2_emotional_safety__pc1": 1.0,
        "Q_repair": 0.7,
        "R1_protective_caretaking": 0.7,
    },
    # 3) Negative affect baseline (collapse anger/anxiety into one axis)
    "AX_negative_affect": {
        "F2_anger_frustration": 1.0,
        "F3_anxiety_worry": 1.0,
        "F1_sadness_grief": 0.6,
    },
    # 4) Explicitness (explicit sexual content only)
    "AX_explicitness": {
        "C_explicit_eroticism": 1.0,
    },
    # 5) Attraction / chemistry (non-explicit romantic charge)
    "AX_attraction": {
        "B1_attraction_chemistry": 1.0,
    },
}

# Build macro-axis columns from df (standardize each component within df, then weighted mean)
def build_macro_axes(df_: pd.DataFrame, macro_axes: dict) -> pd.DataFrame:
    out = df_.copy()
    built = []
    missing_map = {}
    for ax, comps in macro_axes.items():
        vals = None
        wsum = 0.0
        missing = []
        for col, w in comps.items():
            if col not in out.columns:
                missing.append(col)
                continue
            x = pd.to_numeric(out[col], errors="coerce")
            x = (x - x.mean()) / (x.std(ddof=0) + 1e-12)
            if vals is None:
                vals = w * x
            else:
                vals = vals + w * x
            wsum += abs(float(w))
        if missing:
            missing_map[ax] = missing
        if vals is None or wsum == 0:
            out[ax] = np.nan
        else:
            out[ax] = vals / wsum
            built.append(ax)
    return out, built, missing_map

df_macro, BUILT_AXES, AX_MISSING = build_macro_axes(df, MACRO_AXES)

print("Built macro axes:", BUILT_AXES)
if AX_MISSING:
    print("Missing components (check names):", AX_MISSING)

MACRO_PREDICTORS = BUILT_AXES

# Save axis definitions for audit
ax_def = []
for ax, comps in MACRO_AXES.items():
    for col, w in comps.items():
        ax_def.append({"macro_axis": ax, "component": col, "weight": w, "component_present": col in df.columns})
ax_def_df = pd.DataFrame(ax_def)
save_df(ax_def_df, AUDIT_DIR / "macro_axes_definition.csv")
print("✓ Saved macro axes definition to audit_csv/")


Built macro axes: ['AX_status_dominance', 'AX_payoff_safety', 'AX_negative_affect', 'AX_explicitness', 'AX_attraction']
✓ Saved macro axes definition to audit_csv/


**Refinement applied:** `AX_explicitness` now includes **only** explicit erotics, and `AX_attraction` is a separate axis for **chemistry / non-explicit attraction**. The former drama/obstacle axis is removed to keep the macro system tighter (4–5 axes total).

## SECTION 2 — Define predictor sets (CORE gating + recommended scores)

In [7]:
# CORE composites and recommended score choice
core = composite_registry.loc[composite_registry["status_core"]=="CORE", ["composite_key","recommended_score","dimensionality"]].copy()

# Map each composite to the actual column name in df:
# - recommended_score == "sum"  -> use composite_key column (already z-scored)
# - recommended_score == "pc1"  -> use f"{key}__pc1"
# - recommended_score == "atomic_sum" -> treat as exploratory (usually excluded)
def predictor_col(row) -> str:
    k = row["composite_key"]
    rs = str(row.get("recommended_score","sum"))
    if rs == "pc1":
        return f"{k}__pc1"
    return k

core["predictor_col"] = core.apply(predictor_col, axis=1)

# Exclude atomic measures from main analysis
core_main = core.loc[core["recommended_score"]!="atomic_sum"].copy()

# Keep only predictors that exist
core_main = core_main.loc[core_main["predictor_col"].isin(df.columns)].copy()

print("CORE predictors available:", core_main.shape[0])
print(core_main.head(10))

CORE_PREDICTORS = core_main["predictor_col"].tolist()


CORE predictors available: 17
              composite_key recommended_score    dimensionality  \
0         R2_alpha_guarding               sum    UNIDIMENSIONAL   
1                  Q_repair               sum    UNIDIMENSIONAL   
2  M_health_recovery_growth               sum    UNIDIMENSIONAL   
3         I_humor_lightness               sum    UNIDIMENSIONAL   
4        Q_miscommunication               sum    UNIDIMENSIONAL   
5       A2_emotional_safety               pc1  MULTIDIMENSIONAL   
6        L_vices_addictions               sum    UNIDIMENSIONAL   
7  K_professional_intrusion               sum    UNIDIMENSIONAL   
8      C_explicit_eroticism               sum    UNIDIMENSIONAL   
9           S_scene_anchors               sum    UNIDIMENSIONAL   

              predictor_col  
0         R2_alpha_guarding  
1                  Q_repair  
2  M_health_recovery_growth  
3         I_humor_lightness  
4        Q_miscommunication  
5  A2_emotional_safety__pc1  
6        L_vices_addic

## SECTION 3 — Effect sizes + bootstrap sign stability (pilot-appropriate inference)

In [8]:
import numpy as np
import pandas as pd

def zscore(x: pd.Series) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    return (x - x.mean()) / (x.std(ddof=0) + 1e-12)

def ols_beta(X: np.ndarray, y: np.ndarray) -> float:
    # returns coefficient on first column of X (assumes X includes intercept + predictor + optional controls)
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return float(b[1])

def bootstrap_beta(df_: pd.DataFrame, y_col: str, x_col: str, controls: list[str] = None, n_boot: int = 1000, seed: int = 11):
    controls = controls or []
    rng = np.random.default_rng(seed)
    rows = []
    n = len(df_)
    idx = np.arange(n)
    for _ in range(n_boot):
        samp = rng.choice(idx, size=n, replace=True)
        d = df_.iloc[samp]
        y = zscore(d[y_col]).to_numpy()
        x = zscore(d[x_col]).to_numpy()
        X = [np.ones_like(x), x]
        for c in controls:
            X.append(zscore(d[c]).to_numpy())
        X = np.column_stack(X)
        rows.append(ols_beta(X, y))
    betas = np.array(rows, dtype=float)
    beta_hat = float(betas.mean())
    ci_low, ci_high = float(np.quantile(betas, 0.025)), float(np.quantile(betas, 0.975))
    p_pos = float((betas > 0).mean())
    return beta_hat, ci_low, ci_high, p_pos

# Choose outcomes (continuous primary)
OUTCOMES = ["rating_mean", "log_rating_count"]
OUTCOMES = [c for c in OUTCOMES if c in df.columns]

# Controls: keep minimal for small N. Use none by default; optionally add log_rating_count when outcome is rating_mean.
def controls_for(outcome: str) -> list[str]:
    if outcome == "rating_mean" and "log_rating_count" in df.columns:
        return ["log_rating_count"]
    return []

results = []
for outcome in OUTCOMES:
    ctrls = controls_for(outcome)
    for x in CORE_PREDICTORS:
        if x not in df.columns:
            continue
        # drop NA rows
        sub = df[[outcome, x] + ctrls].dropna()
        if len(sub) < 40:
            continue
        beta_hat, ci_low, ci_high, p_pos = bootstrap_beta(sub, outcome, x, controls=ctrls, n_boot=800, seed=11)
        results.append({
            "outcome": outcome,
            "predictor": x,
            "beta_std": beta_hat,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p(beta>0)": p_pos,
            "n": len(sub),
            "controls": ",".join(ctrls) if ctrls else ""
        })

res_df = pd.DataFrame(results).sort_values(["outcome", "p(beta>0)"], ascending=[True, False])
print("Top directional effects (by P(beta>0)):")
display(res_df.head(15))


Top directional effects (by P(beta>0)):


,outcome,predictor,beta_std,ci_low,ci_high,p(beta>0),n,controls
17,log_rating_count,R2_alpha_guarding,0.435101,0.225990,0.596624,1.00000,92,
29,log_rating_count,D_power_wealth_luxury__pc1,0.370879,0.195460,0.545563,1.00000,92,
22,log_rating_count,A2_emotional_safety__pc1,0.323374,0.126439,0.503229,0.99750,92,
18,log_rating_count,Q_repair,0.230164,0.020573,0.405688,0.98500,92,
27,log_rating_count,J_social_support_kin,0.191182,-0.043287,0.426160,0.94875,92,
21,log_rating_count,Q_miscommunication,0.110414,-0.214488,0.440677,0.72000,92,
20,log_rating_count,I_humor_lightness,0.036558,-0.199858,0.229649,0.66250,92,
24,log_rating_count,K_professional_intrusion,0.044871,-0.166195,0.252707,0.64375,92,
19,log_rating_count,M_health_recovery_growth,0.023501,-0.233905,0.204951,0.63625,92,
32,log_rating_count,R1_protective_caretaking,0.023794,-0.259109,0.283841,0.57625,92,


## MACRO-AXES — Level effects (effect sizes + bootstrap sign stability)

Same analysis as CORE predictors, but using macro-axes. This is the cleaner story for the paper.

In [9]:
# ==============================
# MACRO-AXES — Level effects
# ==============================
macro_results = []
for outcome in OUTCOMES:
    # For rating_mean, keep the same control policy as before
    ctrls = controls_for(outcome)
    for x in MACRO_PREDICTORS:
        sub = df_macro[[outcome, x] + ctrls].dropna()
        if len(sub) < 40:
            continue
        beta_hat, ci_low, ci_high, p_pos = bootstrap_beta(sub, outcome, x, controls=ctrls, n_boot=800, seed=21)
        macro_results.append({
            "outcome": outcome,
            "predictor": x,
            "beta_std": beta_hat,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p(beta>0)": p_pos,
            "n": len(sub),
            "controls": ",".join(ctrls) if ctrls else ""
        })

macro_res_df = pd.DataFrame(macro_results).sort_values(["outcome","p(beta>0)"], ascending=[True, False])
display(macro_res_df.groupby("outcome").head(10))

save_df(macro_res_df, AUDIT_DIR / "inference_macro_level_effects.csv")
print("✓ Saved: inference_macro_level_effects.csv")


,outcome,predictor,beta_std,ci_low,ci_high,p(beta>0),n,controls
5,log_rating_count,AX_status_dominance,0.458696,0.280342,0.606545,1.00000,92,
6,log_rating_count,AX_payoff_safety,0.328861,0.144175,0.503018,0.99875,92,
9,log_rating_count,AX_attraction,0.000268,-0.212125,0.216697,0.50875,92,
7,log_rating_count,AX_negative_affect,-0.065355,-0.316826,0.197412,0.31125,92,
8,log_rating_count,AX_explicitness,-0.303509,-0.496111,-0.062834,0.00375,92,
1,rating_mean,AX_payoff_safety,0.214650,-0.001689,0.408554,0.97375,92,log_rating_count
0,rating_mean,AX_status_dominance,0.099134,-0.135261,0.297919,0.82625,92,log_rating_count
4,rating_mean,AX_attraction,-0.003809,-0.206207,0.212489,0.48625,92,log_rating_count
2,rating_mean,AX_negative_affect,-0.130480,-0.296740,0.034180,0.05125,92,log_rating_count
3,rating_mean,AX_explicitness,-0.167243,-0.355893,0.020920,0.04000,92,log_rating_count


✓ Saved: inference_macro_level_effects.csv


## SECTION 4 — Arc tests using exported deltas (end−begin, middle−begin)

This replaces any END-variant creation logic. We test arcs explicitly.

In [10]:
# Identify arc predictors
arc_cols = [c for c in df.columns if c.endswith("__end_minus_begin") or c.endswith("__middle_minus_begin")]
print("Arc predictors:", len(arc_cols))

# Example: test arcs as predictors of rating_mean (pilot-style)
arc_results = []
if "rating_mean" in df.columns and arc_cols:
    for x in arc_cols:
        sub = df[["rating_mean", x, "log_rating_count"]].dropna() if "log_rating_count" in df.columns else df[["rating_mean", x]].dropna()
        if len(sub) < 40:
            continue
        ctrls = ["log_rating_count"] if "log_rating_count" in sub.columns else []
        beta_hat, ci_low, ci_high, p_pos = bootstrap_beta(sub, "rating_mean", x, controls=ctrls, n_boot=800, seed=13)
        arc_results.append({
            "predictor": x,
            "beta_std": beta_hat,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p(beta>0)": p_pos,
            "n": len(sub)
        })

arc_df = pd.DataFrame(arc_results).sort_values("p(beta>0)", ascending=False)
display(arc_df.head(15))


Arc predictors: 52


,predictor,beta_std,ci_low,ci_high,p(beta>0),n
18,F2_anger_frustration__end_minus_begin,0.238412,0.075336,0.412193,0.99500,92
20,F3_anxiety_worry__end_minus_begin,0.189772,0.015351,0.363993,0.98125,92
49,R_jealousy_possessiveness__middle_minus_begin,0.120933,-0.000285,0.261007,0.97250,92
38,O_aesthetics_appearance__end_minus_begin,0.149828,-0.035484,0.315657,0.95000,92
35,K_professional_intrusion__middle_minus_begin,0.182151,-0.130331,0.418555,0.90500,92
47,R2_alpha_guarding__middle_minus_begin,0.088970,-0.042562,0.238130,0.89250,92
3,A2_emotional_safety__middle_minus_begin,0.134413,-0.099997,0.347082,0.87750,92
39,O_aesthetics_appearance__middle_minus_begin,0.135758,-0.109824,0.351356,0.86875,92
12,D_power_wealth_luxury__end_minus_begin,0.091592,-0.073411,0.247448,0.86750,92
34,K_professional_intrusion__end_minus_begin,0.152356,-0.095099,0.388257,0.85750,92


## MACRO-AXES — Arc effects (end−begin, middle−begin)

We aggregate arc deltas into macro-axes by applying the same macro weights to each component's arc delta.

In [11]:
# ==============================
# MACRO-AXES — Arc effects
# ==============================
# Build macro-arc predictors from existing arc_contrasts_sum columns (if present)
if arc_contrasts_sum is None:
    print("arc_contrasts_sum not loaded; skipping macro-arc construction.")
else:
    df_arc = df.merge(arc_contrasts_sum, on="book_id", how="left")

    def build_macro_arc(df_: pd.DataFrame, macro_axes: dict, suffix: str) -> pd.DataFrame:
        out = df_.copy()
        built = []
        for ax, comps in macro_axes.items():
            vals = None
            wsum = 0.0
            for base_col, w in comps.items():
                # Extract base composite key: remove __pc1 suffix if present
                # Arc columns use base key (e.g., "D_power_wealth_luxury__end_minus_begin")
                # not predictor column name (e.g., "D_power_wealth_luxury__pc1__end_minus_begin")
                if base_col.endswith("__pc1"):
                    base_key = base_col[:-5]  # Remove "__pc1"
                else:
                    base_key = base_col
                arc_col = f"{base_key}__{suffix}"
                if arc_col not in out.columns:
                    continue
                x = pd.to_numeric(out[arc_col], errors="coerce")
                # standardize within df for comparability
                x = (x - x.mean()) / (x.std(ddof=0) + 1e-12)
                vals = (w * x) if vals is None else (vals + w * x)
                wsum += abs(float(w))
            if vals is None or wsum == 0:
                out[f"{ax}__{suffix}"] = np.nan
            else:
                out[f"{ax}__{suffix}"] = vals / wsum
                built.append(f"{ax}__{suffix}")
        return out, built

    df_arc_macro = df_arc.copy()
    built_all = []
    for suf in ["end_minus_begin", "middle_minus_begin"]:
        df_arc_macro, built = build_macro_arc(df_arc_macro, MACRO_AXES, suf)
        built_all.extend(built)

    print("Built macro arc columns:", built_all[:10], "..." if len(built_all)>10 else "")

    # Run the same arc analysis (rating_mean by default uses log_rating_count control if available)
    arc_macro_results = []
    if "rating_mean" in df_arc_macro.columns:
        ctrls = ["log_rating_count"] if "log_rating_count" in df_arc_macro.columns else []
        for x in built_all:
            sub = df_arc_macro[["rating_mean", x] + ctrls].dropna()
            if len(sub) < 40:
                continue
            beta_hat, ci_low, ci_high, p_pos = bootstrap_beta(sub, "rating_mean", x, controls=ctrls, n_boot=800, seed=23)
            arc_macro_results.append({
                "outcome": "rating_mean",
                "predictor": x,
                "beta_std": beta_hat,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "p(beta>0)": p_pos,
                "n": len(sub),
                "controls": ",".join(ctrls) if ctrls else ""
            })

    if len(arc_macro_results) == 0:
        print("No macro arc results to analyze (built_all was empty or no valid data).")
        arc_macro_df = pd.DataFrame(columns=["outcome", "predictor", "beta_std", "ci_low", "ci_high", "p(beta>0)", "n", "controls"])
    else:
        arc_macro_df = pd.DataFrame(arc_macro_results).sort_values("p(beta>0)", ascending=False)
        display(arc_macro_df.head(15))
    
    save_df(arc_macro_df, AUDIT_DIR / "inference_macro_arc_effects.csv")
    print("✓ Saved: inference_macro_arc_effects.csv")


Built macro arc columns: [] 
No macro arc results to analyze (built_all was empty or no valid data).
✓ Saved: inference_macro_arc_effects.csv


## SECTION 5 — Predictive usefulness (cross-validated)

Pilot-friendly: compare metadata-only vs +CORE indices.

In [12]:
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

def cv_r2(df_: pd.DataFrame, y_col: str, X_cols: list[str], k: int = 5, seed: int = 11) -> float:
    d = df_[[y_col] + X_cols].dropna()
    if len(d) < 50:
        return np.nan
    y = d[y_col].to_numpy(dtype=float)
    X = d[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
    # standardize X columns
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    preds = np.zeros_like(y)
    for train, test in kf.split(X):
        model = RidgeCV(alphas=np.logspace(-3, 3, 25))
        model.fit(X[train], y[train])
        preds[test] = model.predict(X[test])
    return float(r2_score(y, preds))

# Model sets
meta_features = [c for c in ["log_rating_count"] if c in df.columns]
core_features = CORE_PREDICTORS

pred_rows = []
if "rating_mean" in df.columns:
    pred_rows.append({"outcome":"rating_mean","model":"metadata_only","cv_r2":cv_r2(df,"rating_mean",meta_features)})
    pred_rows.append({"outcome":"rating_mean","model":"metadata+core","cv_r2":cv_r2(df,"rating_mean",meta_features+core_features)})

if "log_rating_count" in df.columns:
    # popularity outcome: use only core features (no need to include itself)
    pred_rows.append({"outcome":"log_rating_count","model":"core_only","cv_r2":cv_r2(df,"log_rating_count",core_features)})

pred_df = pd.DataFrame(pred_rows)
display(pred_df)


,outcome,model,cv_r2
0,rating_mean,metadata_only,0.124947
1,rating_mean,metadata+core,0.049565
2,log_rating_count,core_only,0.033632


## SECTION 6 — Goodreads Metadata Validation (Ratings + Number of Voters)

Two signal channels:
- **avg_rating** (quality perception)
- **n_ratings** (visibility/popularity/market reach)

We treat them differently and create pilot-friendly deliverables: **themes that predict perceived quality** vs **themes that predict mass appeal**.

In [13]:
# ==============================
# SECTION 6.0 — Load Goodreads metadata and align IDs (robust for column variants/casing)
# ==============================
from pathlib import Path
import numpy as np
import pandas as pd
import re

GOODREADS_PATH = Path(os.environ.get("GOODREADS_PATH", DATA_DIR / "processed" / "goodreads.csv"))
# Fallback: user uploaded goodreads.csv in repo root or working dir
if not GOODREADS_PATH.exists():
    alt = Path("goodreads.csv")
    if alt.exists():
        GOODREADS_PATH = alt.resolve()
    else:
        # last resort: look in project root
        alt2 = PROJECT_ROOT / "goodreads.csv"
        if alt2.exists():
            GOODREADS_PATH = alt2
        else:
            raise FileNotFoundError(f"Goodreads CSV not found. Looked at: {GOODREADS_PATH}, {alt}, {alt2}")

gd = pd.read_csv(GOODREADS_PATH)

print("Loaded goodreads:", gd.shape)
print("Columns:", list(gd.columns)[:30], "..." if len(gd.columns)>30 else "")

def _norm_id(x) -> str:
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

# Helper: Look for column match case-insensitively, and with/without s
def _find_col(candidates, cols):
    lookup = {c.lower(): c for c in cols}
    for cand in candidates:
        cand1 = cand.lower()
        if cand1 in lookup:
            return lookup[cand1]
        # Try stripping s for count variants
        if cand1.endswith("s") and cand1[:-1] in lookup:
            return lookup[cand1[:-1]]
        if (cand1 + "s") in lookup:
            return lookup[cand1 + "s"]
    # Try looser match
    for c in cols:
        if any(c.lower() == cand.lower() or c.lower().replace("_","") == cand.lower().replace("_","") for cand in candidates):
            return c
    return None

# Detect ID column to join on
id_candidates = ["book_id", "ID", "goodreads_id", "goodreads_book_id", "gid"]
id_col = _find_col(id_candidates, gd.columns)
if id_col is None:
    raise ValueError(f"No recognizable ID column in goodreads.csv. Tried: {id_candidates}. Available: {list(gd.columns)}")

gd["book_id"] = gd[id_col].map(_norm_id)

# Detect rating + count columns (more robust casing/variants)
rating_candidates = ["avg_rating", "average_rating", "rating_mean", "mean_rating", "score"]
count_candidates = ["n_ratings", "ratings_count", "rating_count", "num_ratings", "votes", "voters", "RatingsCount"]

avg_col = _find_col(rating_candidates, gd.columns)
n_col = _find_col(count_candidates, gd.columns)

# Try common Goodreads export column names if not found
if avg_col is None and "Score" in gd.columns:
    avg_col = "Score"
if n_col is None and "RatingsCount" in gd.columns:
    n_col = "RatingsCount"

if avg_col is None:
    raise ValueError(
        f"No avg rating column found in goodreads.csv. "
        f"Tried {rating_candidates+['Score']}. Available: {list(gd.columns)}"
    )
if n_col is None:
    raise ValueError(
        f"No rating count column found in goodreads.csv. "
        f"Tried {count_candidates+['RatingsCount']}. Available: {list(gd.columns)}"
    )

extra_cols = [c for c in ["title","author","Author","Book Title"] if c in gd.columns]
gd = gd[["book_id", avg_col, n_col] + extra_cols].copy()
gd = gd.rename(columns={avg_col:"avg_rating", n_col:"n_ratings"})

gd["avg_rating"] = pd.to_numeric(gd["avg_rating"], errors="coerce")
gd["n_ratings"] = pd.to_numeric(gd["n_ratings"], errors="coerce")

gd["log_n_ratings"] = np.log1p(gd["n_ratings"])

print("Goodreads aligned preview:")
display(gd.head())
print("Unique IDs:", gd["book_id"].nunique(), "rows:", len(gd))


Loaded goodreads: (97, 14)
Columns: ['ID', 'Author', 'Title', 'URL', 'SeriesName', 'Summary', 'Genres', 'Score', 'RatingsCount', 'ReviewsCount', 'Pages', 'PublishedDate', 'Popularity_ReadingNow', 'Popularity_Wishlisted'] 
Goodreads aligned preview:


,book_id,avg_rating,n_ratings,Author,log_n_ratings
0,35053870,4.07,20705,sarina bowen,9.938179
1,28869598,4.05,10818,sarina bowen,9.289059
2,30627346,3.92,9532,sarina bowen,9.162515
3,17561022,3.82,14878,j. clare,9.607706
4,43728457,3.85,9954,j. clare,9.205830


Unique IDs: 97 rows: 97


In [14]:
# ==============================
# SECTION 6.1 — Basic checks
# ==============================
# Merge Goodreads into the modeling df
df_gd = df.merge(gd[["book_id","avg_rating","n_ratings","log_n_ratings"]], on="book_id", how="left")

missing = df_gd["avg_rating"].isna().mean()
print(f"Missing avg_rating after merge: {missing:.1%}")
missing2 = df_gd["n_ratings"].isna().mean()
print(f"Missing n_ratings after merge: {missing2:.1%}")

# Correlate CORE predictors with avg_rating and log(n_ratings) separately (Pearson + Spearman)
rows = []
for outcome_col in ["avg_rating", "log_n_ratings"]:
    if outcome_col not in df_gd.columns:
        continue
    for x in CORE_PREDICTORS:
        sub = df_gd[[outcome_col, x]].dropna()
        if len(sub) < 20:
            continue
        pearson = float(sub[outcome_col].corr(sub[x], method="pearson"))
        spearman = float(sub[outcome_col].corr(sub[x], method="spearman"))
        rows.append({"outcome": outcome_col, "predictor": x, "pearson_r": pearson, "spearman_r": spearman, "n": len(sub)})

gd_corr = pd.DataFrame(rows).sort_values(["outcome","spearman_r"], ascending=[True, False])
display(gd_corr.groupby("outcome").head(10))

# Tier check: does your tier label differ primarily by avg_rating, n_ratings, or both?
tier_col = "rating_class" if "rating_class" in df_gd.columns else None
if tier_col:
    tier_summary = (df_gd.groupby(tier_col)
                    .agg(n=("book_id","count"),
                         avg_rating_mean=("avg_rating","mean"),
                         avg_rating_sd=("avg_rating","std"),
                         n_ratings_mean=("n_ratings","mean"),
                         log_n_ratings_mean=("log_n_ratings","mean"))
                    .reset_index())
    display(tier_summary)
else:
    print("No 'rating_class' column available for tier comparison in df (from sentence_df).")

# Save outputs
OUTDIR = BUNDLE_DIR / "inference_outputs"
OUTDIR.mkdir(parents=True, exist_ok=True)
gd_corr.to_csv(OUTDIR / "goodreads_index_correlations.csv", index=False)
if tier_col:
    tier_summary.to_csv(OUTDIR / "tier_summary_goodreads_channels.csv", index=False)

print("✓ Saved: goodreads_index_correlations.csv (and tier summary if available) to", OUTDIR)


Missing avg_rating after merge: 0.0%
Missing n_ratings after merge: 0.0%


,outcome,predictor,pearson_r,spearman_r,n
15,avg_rating,R1_protective_caretaking,0.226270,0.252216,92
12,avg_rating,D_power_wealth_luxury__pc1,0.224162,0.236614,92
5,avg_rating,A2_emotional_safety__pc1,0.264254,0.230909,92
0,avg_rating,R2_alpha_guarding,0.240820,0.212424,92
3,avg_rating,I_humor_lightness,0.077627,0.113173,92
11,avg_rating,B1_attraction_chemistry,-0.006941,0.078869,92
4,avg_rating,Q_miscommunication,-0.053250,0.064846,92
1,avg_rating,Q_repair,0.104203,0.037365,92
7,avg_rating,K_professional_intrusion,-0.094292,-0.027012,92
9,avg_rating,S_scene_anchors,-0.056681,-0.048257,92


,rating_class,n,avg_rating_mean,avg_rating_sd,n_ratings_mean,log_n_ratings_mean
0,bad,30,3.774667,0.129368,48121.333333,8.747155
1,good,30,4.215333,0.084842,115577.600000,10.259883
2,mid,32,4.007500,0.051556,44118.812500,9.788559


✓ Saved: goodreads_index_correlations.csv (and tier summary if available) to /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/bundle/inference_outputs


In [15]:
# ==============================
# SECTION 6.2 — Weighted outcomes and noise control
# ==============================
# Motivation: avg_rating from 50 voters is noisier than from 50,000.
# We provide two pilot-friendly alternatives:
#  (A) Weighted regression using weights from n_ratings
#  (B) Bayesian-adjusted rating (shrink low-n toward global mean)

def weighted_simple_beta(d: pd.DataFrame, y: str, x: str, w: str) -> float:
    # WLS for single predictor + intercept, implemented via sqrt(w) scaling
    yv = pd.to_numeric(d[y], errors="coerce").to_numpy(dtype=float)
    xv = pd.to_numeric(d[x], errors="coerce").to_numpy(dtype=float)
    wv = pd.to_numeric(d[w], errors="coerce").to_numpy(dtype=float)
    wv = np.where(np.isfinite(wv) & (wv > 0), wv, 0.0)
    sw = np.sqrt(wv)
    X = np.column_stack([np.ones_like(xv), xv])
    Xw = X * sw[:, None]
    yw = yv * sw
    b, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
    return float(b[1])

# Prepare dataframe
d0 = df_gd[["avg_rating","n_ratings","log_n_ratings"] + CORE_PREDICTORS].dropna().copy()
if len(d0) < 40:
    print("Not enough rows with Goodreads metadata to run weighted analyses.")
else:
    # Bayesian-adjusted rating: (n/(n+k))*r + (k/(n+k))*mu
    mu = float(d0["avg_rating"].mean())
    k = float(np.nanmedian(d0["n_ratings"]))  # prior weight
    d0["avg_rating_bayes"] = (d0["n_ratings"]/(d0["n_ratings"]+k))*d0["avg_rating"] + (k/(d0["n_ratings"]+k))*mu

    # Also compute bayes-adjusted rating for the full merged df_gd (so later blocks can reuse it)
    df_gd = df_gd.copy()
    df_gd["avg_rating_bayes"] = (df_gd["n_ratings"]/(df_gd["n_ratings"]+k))*df_gd["avg_rating"] + (k/(df_gd["n_ratings"]+k))*mu
    print(f"Bayes rating shrinkage params: mu={mu:.4f}, k={k:.1f}")

    # weights: sqrt(n) and log(n) are common; we use sqrt(n) to avoid extreme dominance
    d0["w_sqrt_n"] = np.sqrt(d0["n_ratings"].clip(lower=1.0))
    d0["w_log_n"] = np.log1p(d0["n_ratings"].clip(lower=0.0))

    rows = []
    for x in CORE_PREDICTORS:
        sub = d0[["avg_rating","avg_rating_bayes", x, "w_sqrt_n"]].dropna()
        if len(sub) < 40:
            continue
        # Standardize x for comparability
        sub = sub.copy()
        sub[x] = (sub[x] - sub[x].mean()) / (sub[x].std(ddof=0) + 1e-12)

        # Unweighted simple association
        unweighted_r = float(sub["avg_rating"].corr(sub[x], method="pearson"))

        # Weighted WLS slope (still in standardized-x units)
        beta_w = weighted_simple_beta(sub, "avg_rating", x, "w_sqrt_n")

        # Bayesian-adjusted outcome unweighted correlation
        bayes_r = float(sub["avg_rating_bayes"].corr(sub[x], method="pearson"))

        rows.append({
            "predictor": x,
            "pearson_r_avg": unweighted_r,
            "wls_beta_avg_w_sqrt_n": beta_w,
            "pearson_r_bayes_avg": bayes_r,
            "n": len(sub),
            "mu": mu,
            "k_prior": k
        })

    wtbl = pd.DataFrame(rows).sort_values("wls_beta_avg_w_sqrt_n", ascending=False)
    display(wtbl.head(15))

    wtbl.to_csv(OUTDIR / "goodreads_weighted_quality_table.csv", index=False)
    print("✓ Saved: goodreads_weighted_quality_table.csv to", OUTDIR)

print("Deliverable framing:")
print("- Themes predicting perceived quality: use avg_rating (weighted or bayes-adjusted)")
print("- Themes predicting mass appeal: use log_n_ratings")


Bayes rating shrinkage params: mu=3.9993, k=15659.0


,predictor,pearson_r_avg,wls_beta_avg_w_sqrt_n,pearson_r_bayes_avg,n,mu,k_prior
2,M_health_recovery_growth,0.026816,0.027790,0.095056,92,3.999348,15659.0
5,A2_emotional_safety__pc1,0.264254,0.016980,0.161508,92,3.999348,15659.0
1,Q_repair,0.104203,0.016928,0.032610,92,3.999348,15659.0
15,R1_protective_caretaking,0.226270,0.016693,0.097635,92,3.999348,15659.0
12,D_power_wealth_luxury__pc1,0.224162,0.009863,0.129197,92,3.999348,15659.0
0,R2_alpha_guarding,0.240820,0.007900,0.098527,92,3.999348,15659.0
8,C_explicit_eroticism,-0.270417,0.007317,-0.146804,92,3.999348,15659.0
11,B1_attraction_chemistry,-0.006941,0.004448,0.065319,92,3.999348,15659.0
3,I_humor_lightness,0.077627,0.004272,0.085313,92,3.999348,15659.0
9,S_scene_anchors,-0.056681,0.000823,-0.021206,92,3.999348,15659.0


✓ Saved: goodreads_weighted_quality_table.csv to /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/bundle/inference_outputs
Deliverable framing:
- Themes predicting perceived quality: use avg_rating (weighted or bayes-adjusted)
- Themes predicting mass appeal: use log_n_ratings


## MACRO-AXES — Quality outcome using Bayesian-adjusted Goodreads rating

We treat `avg_rating_bayes` as the primary **perceived quality** outcome (less noisy at low voter counts). We report macro-axis effects both **with** and **without** a `log_n_ratings` control, so we can separate *quality* from *reach*.

In [16]:
# ==============================
# MACRO-AXES — avg_rating_bayes outcome (quality channel)
# ==============================
if "avg_rating_bayes" not in df_gd.columns:
    raise ValueError("avg_rating_bayes not found in df_gd. Run SECTION 6.2 first.")

# Attach bayes rating to macro dataframe
df_macro_gd = df_macro.merge(df_gd[["book_id","avg_rating_bayes","log_n_ratings","n_ratings","avg_rating"]], on="book_id", how="left")

macro_bayes_results = []

# Two variants:
#  (A) no control: "what macro axes predict perceived quality"
#  (B) control log_n_ratings: "what predicts quality beyond reach"
control_sets = [
    ("no_control", []),
    ("control_log_n", ["log_n_ratings"] if "log_n_ratings" in df_macro_gd.columns else []),
]

for tag, ctrls in control_sets:
    for x in MACRO_PREDICTORS:
        sub = df_macro_gd[["avg_rating_bayes", x] + ctrls].dropna()
        if len(sub) < 40:
            continue
        beta_hat, ci_low, ci_high, p_pos = bootstrap_beta(sub, "avg_rating_bayes", x, controls=ctrls, n_boot=800, seed=31)
        macro_bayes_results.append({
            "outcome": "avg_rating_bayes",
            "model": tag,
            "predictor": x,
            "beta_std": beta_hat,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p(beta>0)": p_pos,
            "n": len(sub),
            "controls": ",".join(ctrls) if ctrls else ""
        })

macro_bayes_df = pd.DataFrame(macro_bayes_results).sort_values(["model","p(beta>0)"], ascending=[True, False])
display(macro_bayes_df.groupby("model").head(10))

# Save
save_df(macro_bayes_df, AUDIT_DIR / "inference_macro_level_effects_avg_rating_bayes.csv")
print("✓ Saved: inference_macro_level_effects_avg_rating_bayes.csv")

# Appendix: top 10 per variant
for tag, _ in control_sets:
    top = macro_bayes_df[macro_bayes_df["model"]==tag].sort_values("p(beta>0)", ascending=False).head(10)
    save_df(top, APPENDIX_DIR / f"top10_macro_level_avg_rating_bayes_{tag}.csv")

# Plot (publication-ready)
import matplotlib.pyplot as plt

def coef_dotplot(df_: pd.DataFrame, title: str, path: Path, top_n: int = 10):
    d = df_.copy().sort_values("p(beta>0)", ascending=False).head(top_n)
    d = d.iloc[::-1]
    y = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35*len(d))))
    xerr = np.array([d["beta_std"] - d["ci_low"], d["ci_high"] - d["beta_std"]])
    ax.errorbar(d["beta_std"], y, xerr=xerr, fmt="o", color="C0", capsize=3)
    ax.axvline(0, linestyle="--", linewidth=1, color="gray")
    ax.set_yticks(y)
    ax.set_yticklabels(d["predictor"])
    ax.set_xlabel("Standardized coefficient (bootstrap mean; error bars = 95% CI)")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)

for tag, _ in control_sets:
    d = macro_bayes_df[macro_bayes_df["model"]==tag].copy()
    if len(d):
        coef_dotplot(d, f"Macro-axes → avg_rating_bayes ({tag})", FIG_DIR / f"macro_level_effects_avg_rating_bayes_{tag}.png", top_n=10)

print("✓ Saved bayes-quality plots to:", FIG_DIR)

,outcome,model,predictor,beta_std,ci_low,ci_high,p(beta>0),n,controls
6,avg_rating_bayes,control_log_n,AX_payoff_safety,0.082308,-0.088107,0.252514,0.82250,92,log_n_ratings
9,avg_rating_bayes,control_log_n,AX_attraction,0.065488,-0.103559,0.237503,0.76750,92,log_n_ratings
5,avg_rating_bayes,control_log_n,AX_status_dominance,-0.001081,-0.216991,0.231626,0.48125,92,log_n_ratings
7,avg_rating_bayes,control_log_n,AX_negative_affect,-0.071787,-0.286748,0.149579,0.25875,92,log_n_ratings
8,avg_rating_bayes,control_log_n,AX_explicitness,-0.079296,-0.300670,0.121717,0.23250,92,log_n_ratings
1,avg_rating_bayes,no_control,AX_payoff_safety,0.170547,-0.005118,0.324391,0.97250,92,
0,avg_rating_bayes,no_control,AX_status_dominance,0.133888,-0.081856,0.363908,0.89500,92,
4,avg_rating_bayes,no_control,AX_attraction,0.063922,-0.126887,0.245192,0.75875,92,
2,avg_rating_bayes,no_control,AX_negative_affect,-0.089982,-0.292989,0.159718,0.20000,92,
3,avg_rating_bayes,no_control,AX_explicitness,-0.154219,-0.371766,0.073203,0.08000,92,


✓ Saved: inference_macro_level_effects_avg_rating_bayes.csv
✓ Saved bayes-quality plots to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/inference_runs/inference_20260112_213754__measurement_v5/figures


## SECTION 7 — Joint ridge models (all CORE predictors together)

This addresses collinearity: we fit one shrinkage model with **all CORE predictors jointly** and export standardized coefficients.

In [17]:
from sklearn.linear_model import RidgeCV
import numpy as np
import pandas as pd

OUTDIR = BUNDLE_DIR / "inference_outputs"
OUTDIR.mkdir(parents=True, exist_ok=True)

def fit_ridge_joint(df_: pd.DataFrame, y_col: str, X_cols: list[str], controls: list[str] = None):
    controls = controls or []
    cols = [y_col] + X_cols + controls
    d = df_[cols].dropna().copy()
    if len(d) < 50:
        raise ValueError(f"Not enough rows for ridge fit: {len(d)}")
    y = d[y_col].to_numpy(dtype=float)

    X = d[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)  # standardize predictors

    if controls:
        C = d[controls].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
        C = (C - C.mean(axis=0)) / (C.std(axis=0) + 1e-12)
        X_full = np.column_stack([X, C])
        feature_names = X_cols + controls
    else:
        X_full = X
        feature_names = X_cols

    model = RidgeCV(alphas=np.logspace(-3, 3, 31))
    model.fit(X_full, y)

    coef = pd.Series(model.coef_, index=feature_names, name="coef_std")
    out = pd.DataFrame({
        "feature": coef.index,
        "coef_std": coef.values,
        "abs_coef": np.abs(coef.values),
        "outcome": y_col,
        "n": len(d),
        "alpha": float(model.alpha_)
    }).sort_values("abs_coef", ascending=False)

    return out

# Fit joint ridge for outcomes of interest
if "rating_mean" in df.columns:
    ctrls = ["log_rating_count"] if "log_rating_count" in df.columns else []
    ridge_rm = fit_ridge_joint(df, "rating_mean", CORE_PREDICTORS, controls=ctrls)
    ridge_rm.to_csv(OUTDIR / "ridge_joint_coeffs_rating_mean.csv", index=False)
    display(ridge_rm.head(15))

if "log_rating_count" in df.columns:
    ridge_pop = fit_ridge_joint(df, "log_rating_count", CORE_PREDICTORS, controls=[])
    ridge_pop.to_csv(OUTDIR / "ridge_joint_coeffs_log_rating_count.csv", index=False)
    display(ridge_pop.head(15))

print("✓ Saved ridge joint coefficient tables to:", OUTDIR)


,feature,coef_std,abs_coef,outcome,n,alpha
17,log_rating_count,0.032044,0.032044,rating_mean,92,100.0
7,K_professional_intrusion,-0.018237,0.018237,rating_mean,92,100.0
15,R1_protective_caretaking,0.014492,0.014492,rating_mean,92,100.0
6,L_vices_addictions,-0.011618,0.011618,rating_mean,92,100.0
8,C_explicit_eroticism,-0.010451,0.010451,rating_mean,92,100.0
10,J_social_support_kin,-0.009683,0.009683,rating_mean,92,100.0
13,F2_anger_frustration,-0.009495,0.009495,rating_mean,92,100.0
5,A2_emotional_safety__pc1,0.008683,0.008683,rating_mean,92,100.0
16,H_domestic_nesting__pc1,-0.008482,0.008482,rating_mean,92,100.0
0,R2_alpha_guarding,0.007382,0.007382,rating_mean,92,100.0


,feature,coef_std,abs_coef,outcome,n,alpha
0,R2_alpha_guarding,0.215017,0.215017,log_rating_count,92,158.489319
12,D_power_wealth_luxury__pc1,0.138855,0.138855,log_rating_count,92,158.489319
1,Q_repair,0.127120,0.127120,log_rating_count,92,158.489319
10,J_social_support_kin,0.114213,0.114213,log_rating_count,92,158.489319
5,A2_emotional_safety__pc1,0.109429,0.109429,log_rating_count,92,158.489319
8,C_explicit_eroticism,-0.083996,0.083996,log_rating_count,92,158.489319
2,M_health_recovery_growth,0.067119,0.067119,log_rating_count,92,158.489319
6,L_vices_addictions,-0.063878,0.063878,log_rating_count,92,158.489319
16,H_domestic_nesting__pc1,-0.063016,0.063016,log_rating_count,92,158.489319
4,Q_miscommunication,0.040384,0.040384,log_rating_count,92,158.489319


✓ Saved ridge joint coefficient tables to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/bundle/inference_outputs


## SECTION 8 — Partial correlations (align correlations with controlled models)

For `rating_mean` we compute partial correlations controlling `log_rating_count` (if available). This is a quick, interpretable check that matches the regression setup.

In [18]:
import numpy as np
import pandas as pd

def residualize(y: np.ndarray, C: np.ndarray) -> np.ndarray:
    X = np.column_stack([np.ones(len(y)), C])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return y - X @ b

def partial_corr_table(df_: pd.DataFrame, y_col: str, x_cols: list[str], controls: list[str]) -> pd.DataFrame:
    rows = []
    base_cols = [y_col] + x_cols + controls
    d = df_[base_cols].dropna().copy()
    if len(d) < 30:
        return pd.DataFrame()
    y = d[y_col].to_numpy(dtype=float)

    C = None
    if controls:
        C = d[controls].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)

    for x in x_cols:
        xx = d[x].to_numpy(dtype=float)
        if controls:
            y_res = residualize(y, C)
            x_res = residualize(xx, C)
            r = np.corrcoef(y_res, x_res)[0, 1]
        else:
            r = np.corrcoef(y, xx)[0, 1]
        rows.append({"outcome": y_col, "predictor": x, "partial_r": float(r), "n": len(d), "controls": ",".join(controls) if controls else ""})
    return pd.DataFrame(rows)

if "rating_mean" in df.columns:
    ctrls = ["log_rating_count"] if "log_rating_count" in df.columns else []
    pc_rm = partial_corr_table(df, "rating_mean", CORE_PREDICTORS, controls=ctrls).sort_values("partial_r", ascending=False)
    pc_rm.to_csv(OUTDIR / "partial_corr_rating_mean.csv", index=False)
    display(pc_rm.head(15))

if "log_rating_count" in df.columns:
    pc_pop = partial_corr_table(df, "log_rating_count", CORE_PREDICTORS, controls=[]).sort_values("partial_r", ascending=False)
    pc_pop.to_csv(OUTDIR / "partial_corr_log_rating_count.csv", index=False)
    display(pc_pop.head(15))

print("✓ Saved partial correlation tables to:", OUTDIR)


,outcome,predictor,partial_r,n,controls
15,rating_mean,R1_protective_caretaking,0.244625,92,log_rating_count
5,rating_mean,A2_emotional_safety__pc1,0.151570,92,log_rating_count
12,rating_mean,D_power_wealth_luxury__pc1,0.093269,92,log_rating_count
0,rating_mean,R2_alpha_guarding,0.074750,92,log_rating_count
3,rating_mean,I_humor_lightness,0.064023,92,log_rating_count
2,rating_mean,M_health_recovery_growth,0.010589,92,log_rating_count
1,rating_mean,Q_repair,0.009008,92,log_rating_count
11,rating_mean,B1_attraction_chemistry,-0.009543,92,log_rating_count
9,rating_mean,S_scene_anchors,-0.060684,92,log_rating_count
4,rating_mean,Q_miscommunication,-0.098836,92,log_rating_count


,outcome,predictor,partial_r,n,controls
0,log_rating_count,R2_alpha_guarding,0.440074,92,
12,log_rating_count,D_power_wealth_luxury__pc1,0.354266,92,
5,log_rating_count,A2_emotional_safety__pc1,0.327202,92,
1,log_rating_count,Q_repair,0.235827,92,
10,log_rating_count,J_social_support_kin,0.180377,92,
4,log_rating_count,Q_miscommunication,0.089768,92,
3,log_rating_count,I_humor_lightness,0.047158,92,
2,log_rating_count,M_health_recovery_growth,0.042055,92,
7,log_rating_count,K_professional_intrusion,0.038773,92,
15,log_rating_count,R1_protective_caretaking,0.007193,92,


✓ Saved partial correlation tables to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/bundle/inference_outputs


## SECTION 9 — Repeated cross-validation (reduce small-N randomness)

We repeat K-fold CV over multiple random seeds and summarize mean ± sd.

In [19]:
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

def cv_r2_once(df_: pd.DataFrame, y_col: str, X_cols: list[str], k: int = 5, seed: int = 11) -> float:
    d = df_[[y_col] + X_cols].dropna().copy()
    if len(d) < 50:
        return np.nan
    y = d[y_col].to_numpy(dtype=float)
    X = d[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    preds = np.zeros_like(y)
    for train, test in kf.split(X):
        model = RidgeCV(alphas=np.logspace(-3, 3, 25))
        model.fit(X[train], y[train])
        preds[test] = model.predict(X[test])
    return float(r2_score(y, preds))

def repeated_cv_summary(df_: pd.DataFrame, y_col: str, X_cols: list[str], model_name: str, seeds=range(10, 30), k: int = 5) -> dict:
    scores = []
    for s in seeds:
        scores.append(cv_r2_once(df_, y_col, X_cols, k=k, seed=int(s)))
    scores = np.array(scores, dtype=float)
    scores = scores[~np.isnan(scores)]
    return {
        "outcome": y_col,
        "model": model_name,
        "k": k,
        "n_seeds": int(len(scores)),
        "cv_r2_mean": float(scores.mean()) if len(scores) else np.nan,
        "cv_r2_sd": float(scores.std(ddof=0)) if len(scores) else np.nan,
        "cv_r2_min": float(scores.min()) if len(scores) else np.nan,
        "cv_r2_max": float(scores.max()) if len(scores) else np.nan,
    }

rows = []
meta_features = [c for c in ["log_rating_count"] if c in df.columns]
core_features = CORE_PREDICTORS

if "rating_mean" in df.columns:
    rows.append(repeated_cv_summary(df, "rating_mean", meta_features, "metadata_only"))
    rows.append(repeated_cv_summary(df, "rating_mean", meta_features + core_features, "metadata+core"))

if "log_rating_count" in df.columns:
    rows.append(repeated_cv_summary(df, "log_rating_count", core_features, "core_only"))

cv_rep = pd.DataFrame(rows)
cv_rep.to_csv(OUTDIR / "cv_repeats_summary.csv", index=False)
display(cv_rep)

print("✓ Saved repeated CV summary to:", OUTDIR)


,outcome,model,k,n_seeds,cv_r2_mean,cv_r2_sd,cv_r2_min,cv_r2_max
0,rating_mean,metadata_only,5,20,0.107676,0.031252,-0.000753,0.142212
1,rating_mean,metadata+core,5,20,0.056132,0.041045,-0.030719,0.118063
2,log_rating_count,core_only,5,20,0.049996,0.036743,-0.015930,0.107790


✓ Saved repeated CV summary to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/bundle/inference_outputs


## PUBLICATION PLOTS

Minimal, consistent plots saved to `figures/` (PNG at 300 dpi).

In [20]:
import matplotlib.pyplot as plt

def coef_dotplot(df_: pd.DataFrame, title: str, path: Path, top_n: int = 10):
    d = df_.copy()
    d = d.sort_values("p(beta>0)", ascending=False).head(top_n)
    # Plot in reverse so strongest at top
    d = d.iloc[::-1]
    y = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35*len(d))))
    ax.errorbar(d["beta_std"], y, xerr=[d["beta_std"]-d["ci_low"], d["ci_high"]-d["beta_std"]], fmt="o")
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(d["predictor"])
    ax.set_title(title)
    ax.set_xlabel("Standardized effect (bootstrap mean) with 95% CI")
    fig.tight_layout()
    save_fig(fig, path)
    plt.close(fig)

# Plot macro-axis effects (level) for each outcome
for outcome in OUTCOMES:
    d = macro_res_df[macro_res_df["outcome"]==outcome].copy()
    if len(d):
        coef_dotplot(d, f"Macro-axes level effects → {outcome}", FIG_DIR / f"macro_level_effects_{outcome}.png", top_n=10)

# Plot macro-arc effects for rating_mean if available
if "arc_macro_df" in globals() and isinstance(arc_macro_df, pd.DataFrame) and len(arc_macro_df):
    coef_dotplot(arc_macro_df, "Macro-axes arc effects → rating_mean", FIG_DIR / "macro_arc_effects_rating_mean.png", top_n=12)

print("✓ Saved publication plots to:", FIG_DIR)


✓ Saved publication plots to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/inference_runs/inference_20260112_213754__measurement_v5/figures


## EXPORT — Save key tables to structured folders

Audit tables = full outputs. Appendix tables = top-N summaries for paper appendix.

In [21]:
# Save existing tables (if present) into AUDIT_DIR
if "res_df" in globals() and isinstance(res_df, pd.DataFrame):
    save_df(res_df, AUDIT_DIR / "inference_level_effects_core.csv")
if "arc_df" in globals() and isinstance(arc_df, pd.DataFrame):
    save_df(arc_df, AUDIT_DIR / "inference_arc_effects_core.csv")
if "pred_df" in globals() and isinstance(pred_df, pd.DataFrame):
    save_df(pred_df, AUDIT_DIR / "predictive_cv_single_run.csv")
if "cv_rep" in globals() and isinstance(cv_rep, pd.DataFrame):
    save_df(cv_rep, AUDIT_DIR / "cv_repeats_summary.csv")

# Appendix: top 10 per outcome (core + macro)
if "res_df" in globals() and isinstance(res_df, pd.DataFrame):
    for outcome in res_df["outcome"].unique():
        top = res_df[res_df["outcome"]==outcome].sort_values("p(beta>0)", ascending=False).head(10)
        save_df(top, APPENDIX_DIR / f"top10_core_level_{outcome}.csv")

if "macro_res_df" in globals() and isinstance(macro_res_df, pd.DataFrame):
    for outcome in macro_res_df["outcome"].unique():
        top = macro_res_df[macro_res_df["outcome"]==outcome].sort_values("p(beta>0)", ascending=False).head(10)
        save_df(top, APPENDIX_DIR / f"top10_macro_level_{outcome}.csv")

if "arc_df" in globals() and isinstance(arc_df, pd.DataFrame) and len(arc_df):
    save_df(arc_df.head(25), APPENDIX_DIR / "top25_core_arc_rating_mean.csv")

if "arc_macro_df" in globals() and isinstance(arc_macro_df, pd.DataFrame) and len(arc_macro_df):
    save_df(arc_macro_df.head(25), APPENDIX_DIR / "top25_macro_arc_rating_mean.csv")

print("✓ Exported structured outputs to:", OUTROOT)


✓ Exported structured outputs to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/inference_runs/inference_20260112_213754__measurement_v5
